# clip-grad-norm-pre-step — worked example 1: One clipped training step on a tiny linear model

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`torch.nn.utils.clip_grad_norm_(params, max_norm)` rescales the *global* L2 norm of all gradients down to `max_norm` (only when it exceeds it), preserving direction, and is called AFTER `loss.backward()` but BEFORE `optimizer.step()`. It mutates `.grad` in place and returns the *pre-clip* norm as a tensor. The correct ordering is backward -> clip -> step -> zero_grad.

## Worked solution

**Step 1 - build a model and a forward/backward pass.** We make a 1-layer `nn.Linear`, push a batch through it, take an MSE loss, and call `loss.backward()`. After this, every parameter has a populated `.grad`. Clipping is meaningless before grads exist, so this must come first.

**Step 2 - clip BEFORE stepping.** `clip_grad_norm_(model.parameters(), max_norm)` walks all `.grad` tensors, computes the single global norm `sqrt(sum of all g^2)`, and if that exceeds `max_norm` multiplies every grad by `max_norm / global_norm`. Because it edits `.grad` in place, the optimizer will read the clipped values. It returns the norm measured *before* clipping, which is what you log for monitoring spikes.

**Step 3 - step then zero.** `optimizer.step()` applies the (now clipped) grads. `optimizer.zero_grad()` clears them so the next iteration accumulates fresh. Getting this order wrong (e.g. clipping after step) would do nothing useful.

**Why it works:** clipping the global norm caps the *total* update magnitude without distorting the relative direction of the gradient, which is exactly what stabilizes training when a rare batch produces an exploding gradient.

In [ ]:
import torch.nn as nn
import torch.nn.utils as nn_utils

t.manual_seed(0)

def clipped_step(model, optimizer, x, y, max_norm):
    optimizer.zero_grad()
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    pre_norm = nn_utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
    optimizer.step()
    optimizer.zero_grad()
    return pre_norm.item()

model = nn.Linear(4, 1)
opt = t.optim.SGD(model.parameters(), lr=0.1)
x = t.randn(8, 4)
y = t.randn(8, 1)
pre = clipped_step(model, opt, x, y, max_norm=1.0)
print("pre-clip global norm:", round(pre, 4))